# Day 5 — Cost Tracking & A/B Testing

---

By now you know how much your app costs *in theory*. Today we track it *in reality*:

1. Per-user and per-endpoint cost dashboards
2. **Budget alerts** — get a Slack/email ping when spend exceeds a threshold
3. **A/B testing** two prompts / models / retrievers to know which is better


## 1. Where the money goes

For an LLM app, cost = **input tokens × input price + output tokens × output price + hosting**.

Hosting is usually a rounding error compared to LLM tokens. **Optimize tokens first.**

Common wins:
- Shorter system prompts (Section 6 covered this)
- Cache retrieved chunks — no re-embedding
- Use a smaller model for classification / routing
- Batch embed instead of one-at-a-time


## 2. Track cost per request — SQLite is enough


In [ ]:
import sqlite3
import time

# Prices are examples - keep a config file that reflects current rates
PRICES_USD_PER_MTOK = {
    "openai/gpt-oss-20b":                       {"in": 0.05, "out": 0.20},   # our default
    "openai/gpt-oss-120b":                      {"in": 0.15, "out": 0.60},   # bigger sibling
    "meta-llama/Llama-3.3-70B-Instruct-Turbo":  {"in": 0.88, "out": 0.88},
    "openai/gpt-4o-mini":                       {"in": 0.15, "out": 0.60},
}

conn = sqlite3.connect(":memory:")   # use "usage.db" in production
conn.execute("""CREATE TABLE usage(
    ts INT, user TEXT, endpoint TEXT, model TEXT,
    in_tokens INT, out_tokens INT, cost_usd REAL
)""")


def log_call(user, endpoint, model, in_tok, out_tok):
    p = PRICES_USD_PER_MTOK[model]
    cost = (in_tok / 1e6) * p["in"] + (out_tok / 1e6) * p["out"]
    conn.execute(
        "INSERT INTO usage VALUES (?, ?, ?, ?, ?, ?, ?)",
        (int(time.time()), user, endpoint, model, in_tok, out_tok, cost),
    )
    return cost


log_call("alice", "/ask", "openai/gpt-oss-20b",  500, 200)
log_call("bob",   "/ask", "openai/gpt-oss-120b", 500, 200)
log_call("alice", "/ask", "openai/gpt-4o-mini", 500, 200)

for row in conn.execute("SELECT user, model, ROUND(cost_usd, 5) FROM usage"):
    print(row)


## 3. Cheap dashboards — no Grafana required

For a fresher project, a `/metrics` endpoint that returns aggregates as JSON is enough. Wire it into any lightweight dashboard tool (Metabase, Streamlit, or just `curl | jq`).


In [ ]:
def cost_by_user_today() -> list[dict]:
    rows = conn.execute("""
        SELECT user,
               SUM(in_tokens + out_tokens) AS tokens,
               ROUND(SUM(cost_usd), 4)     AS usd
        FROM usage
        WHERE ts >= strftime('%s', 'now', 'start of day')
        GROUP BY user
        ORDER BY usd DESC
    """).fetchall()
    return [{"user": u, "tokens": t, "usd": c} for u, t, c in rows]

print(cost_by_user_today())


## 4. Budget alerts — the one-time-fee prevention

Before shipping anything to real users, add per-user daily caps:

```python
DAILY_CAP_USD = 1.0

def check_budget(user_id: str) -> None:
    row = conn.execute(
        "SELECT COALESCE(SUM(cost_usd),0) FROM usage "
        "WHERE user=? AND ts >= strftime('%s','now','start of day')",
        (user_id,),
    ).fetchone()
    if row[0] > DAILY_CAP_USD:
        raise HTTPException(429, f"Daily budget exceeded ({row[0]:.2f} USD)")
```

Add it as a FastAPI dependency on any expensive endpoint. Users get a clean 429 instead of you getting a $500 bill.


## 5. Global spend alert (Slack)

For your own account-level spend, wire a nightly cron:


In [ ]:
import httpx, os, datetime

def slack(text: str) -> None:
    url = os.getenv("SLACK_WEBHOOK_URL")
    if url:
        httpx.post(url, json={"text": text})

total_today = conn.execute(
    "SELECT COALESCE(SUM(cost_usd),0) FROM usage "
    "WHERE ts >= strftime('%s','now','start of day')"
).fetchone()[0]

if total_today > 5.0:                # your global daily threshold
    slack(f":money_with_wings: Daily LLM spend: ${total_today:.2f}")
print(f"today total: ${total_today:.4f}")


## 6. A/B testing — two prompts, honest comparison

You want to know: **is the new prompt actually better?** The answer is not "read the outputs and vibe." It's a controlled test.

**The simple recipe:**

1. Give each incoming request a `variant` ("A" or "B") — assign randomly and stick to it per user.
2. Log the variant with every trace.
3. After N requests per variant, compare a metric (thumbs-up rate, latency, cost).

Frameworks (**Statsig**, **PostHog**, **Split.io**) do this at scale. For learning, a random flag + your usage table is enough.


In [ ]:
import random

def assign_variant(user_id: str) -> str:
    # deterministic per-user so a user always sees the same variant
    return "B" if hash(user_id) % 2 else "A"

PROMPTS = {
    "A": "Answer the user's question.",
    "B": "Answer the user's question in one paragraph, citing sources.",
}

# When logging, add a 'variant' column and record which prompt was used.
# When analyzing, GROUP BY variant and compare thumbs-up rate + latency.


**Do not** ship a "big prompt change" without A/B testing it. A prompt tweak that "feels better" often reduces success rate on the long tail of real questions.


## 7. When to stop optimizing

Cost optimization is a rabbit hole. Two stopping rules:

- **Cost per active user < 5% of revenue per active user.** If you're not there yet, keep optimizing.
- **p95 latency < your UX threshold** (typical: 2–3 s for chat, 5–10 s for RAG). If you're comfortably under, spend engineering effort elsewhere.

Ship first, optimize the top 3 endpoints. Do not optimize the endpoint that runs once a week.


## Recap

- **Cost = tokens × price.** Track it per user, per endpoint, per model.
- SQLite + a `/metrics` endpoint is enough of a dashboard for fresher projects.
- **Set per-user daily caps.** Cheap insurance against runaway bills.
- **A/B test** prompt / model / retriever changes — don't ship on vibes.
- Stop optimizing when cost < 5% of revenue and latency < user threshold.
- **Next class:** put everything together — deploy the Section 6 RAG chatbot for real.
